# Lab 10 — Supervisor-worker from scratch

Build a 3-agent system using only the Lab 01-03 machinery.

- **Supervisor** — coordinator; sees the workers as tools.
- **Researcher** — Lab 03's web_search + fetch_page; returns findings and citations.
- **Writer** — prompt-only; turns findings into prose; preserves citations verbatim.

No frameworks. No multi-agent libraries. The supervisor's "tools" are
Python functions that internally run their own agent loop — dispatched
through the same `chat_with_tools` contract from Lab 02.

> ⏱ Run time: ~100-130 min including reading.
> 📖 The lab notebook is the assembly. The *why* behind each decision lives in
> [`concepts/multi-agent/supervisor-worker-pattern.md`](../../concepts/multi-agent/supervisor-worker-pattern.md)
> and [`concepts/multi-agent/handoffs-and-shared-state.md`](../../concepts/multi-agent/handoffs-and-shared-state.md).
> Read those first.

## Step 0: Setup

Same setup as Labs 01-03. The provider-agnostic `chat_with_tools` client
is the same one — we're not introducing a new abstraction.

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import time
import warnings
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

# Walk up to find .env at the repo root
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)

PROVIDER = "openai"   # or "anthropic"
MODEL = {
    "openai": "gpt-4o-mini",
    "anthropic": "claude-haiku-4-5-20251001",
}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


**Sample output:**

```
Using openai / gpt-4o-mini
```

## The chat client — unchanged from Lab 02

Provider-agnostic `chat_with_tools` returning an `AssistantMessage`
with `content` and `tool_calls`. The supervisor, the researcher, and
the writer will all use this same client. It's not a multi-agent
abstraction — it's the same single-agent client we've used since Lab 01.

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(
    messages: list[dict],
    tools: list[dict] | None = None,
    tool_choice: str = "auto",
) -> AssistantMessage:
    """Send messages, optionally with tools; return structured assistant response."""
    if PROVIDER == "openai":
        from openai import OpenAI

        resp = OpenAI().chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice if tools else None,
            temperature=0,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(
                    id=tc.id,
                    name=tc.function.name,
                    arguments=json.loads(tc.function.arguments),
                )
                for tc in (msg.tool_calls or [])
            ],
        )

    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {
                "name": t["function"]["name"],
                "description": t["function"]["description"],
                "input_schema": t["function"]["parameters"],
            }
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL,
            system=system,
            messages=non_system,
            tools=anth_tools or None,
            max_tokens=2048,
            temperature=0,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [
            ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
            for b in resp.content
            if getattr(b, "type", None) == "tool_use"
        ]
        return AssistantMessage(content=text or None, tool_calls=tcs)

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


## Step 1: The researcher worker

Lab 03's `web_search` and `fetch_page` verbatim — structured-error
envelope on every failure mode, ddgs + requests + bs4. The researcher
worker runs its own Lab 03-style agent loop with these tools, tracking
citations *by the loop* (not by the LLM) on successful `fetch_page` calls.

The researcher's input is `{"question": str}`. Its output is
`{"status": "ok" | ..., "findings": str, "citations": [...]}`.

Skip this cell quickly if you've done Lab 03; the code is the same.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException

RecencyType = Literal["any", "day", "week", "month", "year"]
_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}


def web_search(
    query: str, recency: RecencyType = "any", max_results: int = 8,
) -> dict:
    """Search the web (ddgs-backed). Returns structured results or error."""
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(), region="us-en", safesearch="moderate",
                timelimit=_RECENCY_MAP.get(recency),
                max_results=max_results, backend="auto",
            )
    except RatelimitException as e:
        return {"status": "error", "kind": "rate_limit", "detail": str(e)}
    except TimeoutException as e:
        return {"status": "error", "kind": "timeout", "detail": str(e)}
    except DDGSException as e:
        return {"status": "error", "kind": "other", "detail": str(e)}
    except Exception as e:
        return {"status": "error", "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}

    if not raw:
        return {"status": "empty", "query": query, "detail": "no results returned"}

    return {
        "status": "ok",
        "results": [
            {
                "title": (r.get("title") or "").strip(),
                "url": (r.get("href") or "").strip(),
                "snippet": (r.get("body") or "").strip(),
            }
            for r in raw if r.get("href")
        ][:max_results],
    }


import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

USER_AGENT = (
    "AgenticAIEngineer-CourseLab/0.1 "
    "(https://github.com/MHHamdan/Agentic-AI-Engineer) "
    "Mozilla/5.0 (compatible)"
)
PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit",
    "register to read",
]


def fetch_page(url: str, max_chars: int = 8000) -> dict:
    """Fetch a URL; return cleaned text or structured error."""
    if not url or not url.startswith(("http://", "https://")):
        return {"status": "error", "url": url, "kind": "other", "detail": "invalid url"}
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT},
                            timeout=15, allow_redirects=True)
    except requests.Timeout:
        return {"status": "error", "url": url, "kind": "timeout",
                "detail": "request timed out after 15s"}
    except requests.RequestException as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}

    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return {"status": "error", "url": url, "kind": kind,
                "detail": f"HTTP {resp.status_code}"}
    if 500 <= resp.status_code < 600:
        return {"status": "error", "url": url, "kind": "http_5xx",
                "detail": f"HTTP {resp.status_code}"}

    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return {"status": "error", "url": url, "kind": "parse",
                "detail": f"{type(e).__name__}: {e}"}

    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()

    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()

    if any(m in text[:5000].lower() for m in PAYWALL_MARKERS):
        return {"status": "error", "url": url, "kind": "paywall",
                "detail": "paywall markers detected in body"}

    if len(text) > max_chars:
        return {"status": "too_long", "url": url, "title": title,
                "text": text[:max_chars], "total_chars": len(text)}
    return {"status": "ok", "url": url, "title": title, "text": text}


### The researcher's agent loop

A Lab 03-style loop, repackaged as a function that takes a `question`
string and returns a structured result. The loop logic is *unchanged*
from Lab 03's solution — it's just wrapped in a function call so the
supervisor can invoke it as a tool.

Note: the function returns a dict, not a string. That's rule 1 of
handoff hygiene (structured payloads, not free text) from the
[handoffs concept page](../../concepts/multi-agent/handoffs-and-shared-state.md#rule-1-handoffs-carry-structured-payloads-not-free-text).

In [ ]:
RESEARCHER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": (
                "Search the web. Returns up to max_results items with title, url, "
                "and a short snippet. Use this to triage which pages to fetch."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "3-8 specific words."},
                    "recency": {
                        "type": "string",
                        "enum": ["any", "day", "week", "month", "year"],
                        "description": "'any' for stable topics, 'week'/'month' for news.",
                    },
                    "max_results": {"type": "integer", "description": "1-10, default 8."},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_page",
            "description": (
                "Fetch the full content of a single URL. Use after web_search to "
                "read the most relevant pages. Returns cleaned text or structured error."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string"},
                    "max_chars": {"type": "integer"},
                },
                "required": ["url"],
            },
        },
    },
]


def _researcher_execute_tool(name: str, args: dict) -> dict:
    if name == "web_search":
        return web_search(
            query=args["query"],
            recency=args.get("recency", "any"),
            max_results=args.get("max_results", 8),
        )
    if name == "fetch_page":
        return fetch_page(url=args["url"], max_chars=args.get("max_chars", 8000))
    return {"status": "error", "kind": "other", "detail": f"unknown tool: {name}"}


def _action_hash(name: str, args: dict) -> str:
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


WORKER_MAX_STEPS = 8


RESEARCHER_SYSTEM_PROMPT = """You are a research worker. Answer the given question
by searching the web and reading 1-2 pages. When you have grounded findings,
respond with a plain text summary — no further tool calls — and the loop will
return your findings to the supervisor along with the citations that were
recorded structurally.

Strategy:
1. web_search with 3-8 specific words from the question.
2. If snippets answer the question, summarize. If not, fetch_page on the 1-2 most
   relevant URLs and summarize from their full text.
3. Never invent citations. Only pages you actually fetched get cited (the loop
   tracks this; do not list URLs you only saw in snippets).
4. If the corpus genuinely doesn't have the answer, say so plainly.
"""


def researcher_agent(question: str, verbose: bool = False) -> dict:
    """Worker: run a Lab 03-style loop. Return {status, findings, citations}."""
    messages: list[dict] = [
        {"role": "system", "content": RESEARCHER_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()

    for step in range(1, WORKER_MAX_STEPS + 1):
        if verbose:
            print(f"    [researcher step {step}]")

        msg = chat_with_tools(messages, tools=RESEARCHER_TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        # Terminal: no tool calls → researcher produced findings
        if not msg.tool_calls:
            return {
                "status": "ok",
                "findings": msg.content or "",
                "citations": citations,
                "steps": step,
            }

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {
                    "status": "error", "kind": "repeated_action",
                    "detail": f"You already called {tc.name} with these arguments.",
                }
            else:
                seen_actions.add(ah)
                tool_result = _researcher_execute_tool(tc.name, tc.arguments)
                # Citations: only successful fetch_page calls
                if tc.name == "fetch_page" and tool_result.get("status") in ("ok", "too_long"):
                    citations.append({
                        "url": tool_result["url"],
                        "title": tool_result.get("title", ""),
                    })

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(tool_result)[:4000],
            })

    # Researcher hit its step cap — return partial findings, not exception.
    # The supervisor reads this status and decides what to do.
    return {
        "status": "step_cap",
        "findings": (
            "Research did not complete within the step budget. "
            f"I fetched {len(citations)} page(s) but did not produce a final summary."
        ),
        "citations": citations,
        "steps": WORKER_MAX_STEPS,
    }


**Quick check — researcher alone:**

```python
result = researcher_agent(
    "What is the Model Context Protocol (MCP)?", verbose=True
)
print(result["findings"][:200])
print(f"Citations: {len(result['citations'])}")
```

This call hits live web; expect 3-10 seconds and 1-3 fetches.

## Step 2: The writer worker

The writer has no tools. Its job is to turn a structured brief
(findings + citations) into ~150-word prose that *preserves the
citations*. No external calls; one LLM call. The agent-loop machinery
isn't strictly needed for a prompt-only worker, but we use it anyway
for uniformity — the supervisor sees both workers through the same
tool-calling lens.

In [ ]:
WRITER_SYSTEM_PROMPT = """You are a writer worker. You receive a brief containing
findings and a list of citations. Produce ~150 words of clean prose that:

1. States the findings accurately. Do not invent claims not present in the brief.
2. Preserves the citations. Reference them inline using [1], [2], etc. matching
   the order in the citation list. Then list the citations at the end as:

       [1] Title — URL
       [2] Title — URL

3. If the brief is insufficient (e.g., findings say "couldn't determine X"),
   honestly say so in the prose. Do not paper over gaps.

If the brief is genuinely empty or malformed, return:
   {"status": "needs_more_research", "missing": [...]}
"""


def writer_agent(brief: dict, verbose: bool = False) -> dict:
    """Worker: produce prose from a brief. Return {status, answer}."""
    if not brief or not brief.get("findings"):
        return {
            "status": "needs_more_research",
            "missing": ["empty or malformed brief"],
        }

    # Render the brief as the user prompt
    citations_block = "\n".join(
        f"[{i + 1}] {c.get('title', '')} — {c['url']}"
        for i, c in enumerate(brief.get("citations", []))
    ) or "(no citations)"
    user_prompt = (
        f"FINDINGS:\n{brief['findings']}\n\n"
        f"CITATIONS:\n{citations_block}\n\n"
        f"Write the ~150-word answer following the rules in your system prompt."
    )

    msg = chat_with_tools([
        {"role": "system", "content": WRITER_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])
    if verbose:
        print(f"    [writer produced {len(msg.content or '')} chars]")
    return {
        "status": "ok",
        "answer": msg.content or "",
    }


**Quick check — writer alone with a fabricated brief:**

```python
fake_brief = {
    "findings": "MCP is a protocol for tool integration with LLMs, introduced by Anthropic in 2024.",
    "citations": [
        {"url": "https://example.com/mcp", "title": "MCP Overview"},
    ],
}
out = writer_agent(fake_brief, verbose=True)
print(out["answer"])
```

## Step 3: The supervisor

The supervisor's "tools" are the two worker functions. We expose them
to the supervisor through the same Lab 02 tool-calling contract:
Pydantic schemas (with `extra="forbid"`), terse names, and
**negative-guidance descriptions** that tell the supervisor when *not*
to use each worker.

The supervisor's job is route + synthesize. It calls the researcher
when fresh information is needed, then the writer to compose the final
answer. It is *not* the supervisor's job to do the research or write
the prose — those are workers' jobs. The supervisor's system prompt
makes this explicit.

`SUPERVISOR_MAX_STEPS = 6` — smaller than the workers' `WORKER_MAX_STEPS = 8`,
because routing is shallower work than research.

In [ ]:
SUPERVISOR_MAX_STEPS = 6


# ── Worker tool schemas (strict mode) ─────────────────────────────────────

class StrictModel(BaseModel):
    """Pydantic base with extra='forbid'. Same pattern as Lab 02."""
    model_config = ConfigDict(extra="forbid")


class CallResearcherArgs(StrictModel):
    question: str = Field(
        description=(
            "The question the researcher should answer. Should be a clear, "
            "self-contained question — do not pass partial phrases or topic "
            "names. The researcher will search and fetch pages on this exact "
            "question."
        )
    )


class CallWriterArgs(StrictModel):
    findings: str = Field(
        description=(
            "The research findings that the writer should turn into prose. "
            "Should be a coherent factual summary, typically obtained from "
            "a prior call_researcher result."
        )
    )
    citations: list[dict] = Field(
        default_factory=list,
        description=(
            "List of citation objects from the researcher. Each is "
            "{url: str, title: str}. The writer will preserve these inline "
            "in the prose."
        ),
    )


def _call_researcher_tool(args: CallResearcherArgs) -> dict:
    return researcher_agent(args.question)


def _call_writer_tool(args: CallWriterArgs) -> dict:
    return writer_agent({
        "findings": args.findings,
        "citations": args.citations,
    })


SUPERVISOR_TOOLS = {
    "call_researcher": (
        _call_researcher_tool,
        CallResearcherArgs,
        # Description seen by the supervisor LLM. Includes negative guidance.
        "Send a question to the researcher worker. The researcher will search "
        "the web and read 1-2 pages, returning {findings, citations}. "
        "Use this when fresh information is needed to answer the user's task. "
        "Do NOT use this if the question can be answered from this conversation "
        "history alone; just call call_writer directly. Do NOT call this more "
        "than once with the same question — refine the question if you need more.",
    ),
    "call_writer": (
        _call_writer_tool,
        CallWriterArgs,
        "Send research findings + citations to the writer worker. The writer "
        "produces ~150 words of prose with citations preserved inline. "
        "Call this once you have enough information from the researcher to "
        "answer the user's task. Do NOT skip this for tasks that require "
        "a written answer — calling call_writer is how the supervisor produces "
        "the final prose.",
    ),
}


def _supervisor_make_schemas() -> list[dict]:
    return [
        {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": args_model.model_json_schema(),
            },
        }
        for name, (_fn, args_model, description) in SUPERVISOR_TOOLS.items()
    ]


def _supervisor_execute_tool(call: ToolCall) -> dict:
    if call.name not in SUPERVISOR_TOOLS:
        return {
            "status": "error", "kind": "unknown_worker",
            "tool": call.name, "available": list(SUPERVISOR_TOOLS),
        }
    fn, args_model, _ = SUPERVISOR_TOOLS[call.name]
    try:
        return fn(args_model.model_validate(call.arguments))
    except Exception as e:
        return {
            "status": "error", "kind": "supervisor_dispatch_error",
            "detail": f"{type(e).__name__}: {e}",
        }


SUPERVISOR_SYSTEM_PROMPT = """You are a supervisor agent coordinating two workers:
a researcher (with web tools) and a writer (no tools, prose only).

Your job:
1. For tasks needing fresh information, call call_researcher with a clear question.
2. Take the researcher's findings + citations and call call_writer to produce prose.
3. Return the writer's answer to the user. Do NOT rewrite the prose yourself.

CRITICAL: The citations from the researcher MUST be preserved in the final answer.
The writer is responsible for inlining them; you are responsible for passing them
through unchanged when you call call_writer.

If a worker returns status='step_cap' or 'error', read the detail and decide what
to do. Do not silently ignore worker errors.

Do not loop. Do not call the same worker twice with the same arguments — the
system will refuse repeats.
"""


def supervisor_agent(task: str, verbose: bool = True) -> dict:
    """Run the supervisor's agent loop. Returns the final answer + trace metadata."""
    messages: list[dict] = [
        {"role": "system", "content": SUPERVISOR_SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    seen_actions: set[str] = set()
    trace: list[dict] = []
    schemas = _supervisor_make_schemas()

    for step in range(1, SUPERVISOR_MAX_STEPS + 1):
        if verbose:
            print(f"\n── Supervisor step {step} ──")

        msg = chat_with_tools(messages, tools=schemas)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        # Terminal: no tool calls → supervisor is returning the final answer
        if not msg.tool_calls:
            if verbose:
                print(f"  ◆ FINAL: {(msg.content or '')[:120]}...")
            return {
                "answer": msg.content or "",
                "trace": trace,
                "steps": step,
                "stopped_reason": "supervisor_returned",
            }

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {
                    "status": "error", "kind": "repeated_action",
                    "detail": f"You already called {tc.name} with these arguments. "
                              f"Either move to the next step or change the arguments.",
                }
                if verbose:
                    print(f"  ✗ {tc.name}(...) [REPEATED — refused]")
            else:
                seen_actions.add(ah)
                if verbose:
                    print(f"  → {tc.name}(...)")
                tool_result = _supervisor_execute_tool(tc)
                if verbose:
                    summary = (tool_result.get("answer") or
                               tool_result.get("findings") or
                               str(tool_result))[:80]
                    print(f"  ← status={tool_result.get('status', '?')}: {summary}...")

            trace.append({
                "step": step,
                "tool": tc.name,
                "args": tc.arguments,
                "result_status": tool_result.get("status"),
            })
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(tool_result)[:6000],
            })

    return {
        "answer": "[Supervisor hit step cap without producing a final answer]",
        "trace": trace,
        "steps": SUPERVISOR_MAX_STEPS,
        "stopped_reason": "step_cap",
    }


## Step 4: Run the supervisor end-to-end

A real task. The supervisor should call researcher once, then writer
once, then return. Verbose trace shows the full trajectory.

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP) "
    "and write a 150-word summary."
)
result = supervisor_agent(task)
print("\n" + "=" * 70)
print("FINAL ANSWER:")
print("=" * 70)
print(result["answer"])
print()
print(f"Steps: {result['steps']}, stopped: {result['stopped_reason']}")
print(f"Trace summary: {len(result['trace'])} tool calls")


**Sample output (LLM responses will vary; trajectory should be stable):**

```
── Supervisor step 1 ──
  → call_researcher(...)
    [researcher step 1]
    [researcher step 2]
    [researcher step 3]
  ← status=ok: MCP (Model Context Protocol) is an open standard introduced by ...

── Supervisor step 2 ──
  → call_writer(...)
  ← status=ok: The Model Context Protocol (MCP) is an open standard introduced by Anthropic...

── Supervisor step 3 ──
  ◆ FINAL: The Model Context Protocol (MCP) is an open standard introduced by Anthropic...

======================================================================
FINAL ANSWER:
======================================================================
The Model Context Protocol (MCP) is an open standard introduced by Anthropic
in late 2024 to standardize how LLM applications integrate with external
data sources and tools [1]. The protocol defines a client-server architecture
where MCP clients (typically LLM applications) communicate with MCP servers
that expose resources, tools, and prompts [2]. ...

[1] Introducing the Model Context Protocol — https://anthropic.com/news/mcp
[2] MCP Specification — https://modelcontextprotocol.io/docs

Steps: 3, stopped: supervisor_returned
Trace summary: 2 tool calls
```

The supervisor took 3 of its 6 allowed steps — one to call the researcher,
one to call the writer, and one to return. The researcher's internal
trajectory was 3 steps (its own budget of 8). The writer doesn't loop.

## Step 5: Failure-mode walkthrough

Three failure modes worth seeing in action.

### Failure mode 1: Supervisor over-calling the researcher

What if the supervisor tries to call `call_researcher` twice with the
same question? The action-hash dedup catches it and returns a structured
error. The supervisor reads the error and either moves on or changes its
approach. This is the same pattern from Lab 03's repeated-action detection,
just lifted up to the supervisor level.

To see this happen naturally we'd need an adversarial prompt — but the
mechanism is straightforward. The dedup check fires inside `supervisor_agent`
before the worker is invoked, so the cost of a repeated call is one LLM
call (the supervisor's bad routing decision), not a redundant worker run.

In [ ]:
# Demonstrate the dedup mechanism directly
sig_a = _action_hash("call_researcher", {"question": "What is MCP?"})
sig_b = _action_hash("call_researcher", {"question": "What is MCP?"})
sig_c = _action_hash("call_researcher", {"question": "What is A2A?"})
print(f"Same question  → same hash: {sig_a == sig_b}")
print(f"Different q    → different: {sig_a != sig_c}")


**Sample output:**

```
Same question  → same hash: True
Different q    → different: True
```

### Failure mode 2: Worker hits its step cap

If the researcher exhausts `WORKER_MAX_STEPS` without producing findings,
it returns `{"status": "step_cap", "findings": "...did not complete...",
"citations": [...]}`. The supervisor receives this as a normal tool
result — *not* as an exception. The supervisor's system prompt explicitly
says "do not silently ignore worker errors," so a well-prompted supervisor
should surface the partial result to the user rather than calling researcher
again with the same question (which would be caught by dedup anyway).

In [ ]:
# Show what the supervisor receives when a worker step-caps
# (this is a fabricated example showing the envelope shape)
fake_researcher_result = {
    "status": "step_cap",
    "findings": "Research did not complete within the step budget. I fetched 1 page(s) but did not produce a final summary.",
    "citations": [{"url": "https://example.com/...", "title": "..."}],
    "steps": 8,
}
print(json.dumps(fake_researcher_result, indent=2))


The supervisor reads `status == "step_cap"`, sees the partial
citations, and decides whether to call the writer with the partial
result or to surface the limitation to the user.

### Failure mode 3: Supervisor calls a nonexistent worker

If the supervisor tries to call a worker that doesn't exist (model
hallucinating a tool name), the dispatcher returns a structured
`unknown_worker` error. The supervisor reads it and tries again.
This is the Lab 02 unknown-tool envelope, lifted to the supervisor
level — same structure, same recovery path.

In [ ]:
# Simulate a hallucinated worker name
fake_call = ToolCall(id="fake_id", name="call_critic", arguments={"text": "..."})
result = _supervisor_execute_tool(fake_call)
print(json.dumps(result, indent=2))


**Sample output:**

```json
{
  "status": "error",
  "kind": "unknown_worker",
  "tool": "call_critic",
  "available": ["call_researcher", "call_writer"]
}
```

The `available` field is doing real work: it tells the LLM exactly
what *is* available, which is more useful than "tool not found" alone.

## Step 6 (stretch): Add a critic worker

A third worker that reviews the writer's output and either approves
or requests revisions. The supervisor invokes it after `call_writer`
and before returning the final answer. This is a step toward the
**agent debate** pattern (a future Path 03 module), but kept simple
here: one-shot critique, no iterative revision.

The critic checks two things:

1. **Citation preservation** — did the writer use the citations correctly?
2. **Grounding** — does every factual claim in the prose appear in the findings?

If either check fails, the critic returns `{"status": "needs_revision", ...}`
and the supervisor can call the writer again with a revised brief.

For brevity, this section sketches the pattern but does not extend
the supervisor loop. See `solution/lab.ipynb` (forthcoming) for the
full implementation.

In [ ]:
CRITIC_SYSTEM_PROMPT = """You are a critic worker. You receive a draft answer and
the original brief (findings + citations). Check two things:

1. Citation preservation: Are all citations from the brief actually used in the draft?
2. Grounding: Does every factual claim in the draft appear in or follow from the findings?

Respond with one of:
- {"status": "ok"} if both checks pass.
- {"status": "needs_revision", "issues": [...]} if either fails.

Be strict but fair. If a claim is generally true but not in the findings, flag it.
"""


def critic_agent(draft: str, brief: dict) -> dict:
    """Worker: review the draft. Return {status, issues}."""
    user_prompt = (
        f"BRIEF FINDINGS:\n{brief.get('findings', '')}\n\n"
        f"BRIEF CITATIONS:\n{json.dumps(brief.get('citations', []), indent=2)}\n\n"
        f"DRAFT TO REVIEW:\n{draft}\n\n"
        f"Apply the checks from your system prompt. Respond with JSON only."
    )
    msg = chat_with_tools([
        {"role": "system", "content": CRITIC_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ])
    # Best-effort JSON parse; in production wrap with stricter validation
    raw = (msg.content or "").strip()
    # Strip code fences if model added them
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"status": "ok", "_raw": raw[:200]}


Wiring `critic_agent` into the supervisor's tool registry is one
line: add `"call_critic": (critic_tool, CallCriticArgs, "Review the
writer's draft for citation preservation and grounding...")` to
`SUPERVISOR_TOOLS`. The supervisor's prompt would also need updating
to invoke it. Both are exercises for the solution batch.

## What you just built

A 3-agent supervisor-worker system in ~250 lines of Python, with:

- Three independent agent loops (supervisor + researcher + writer), each with their own system prompt, tools, and step cap.
- Workers exposed to the supervisor through the same Lab 02 tool-calling contract — workers aren't a new abstraction, they're tools whose handlers happen to run agent loops.
- Structured handoff envelopes (`{status, findings, citations}`, `{status, answer}`) — not free text.
- Citation tracking that survives the worker → supervisor → final-answer path, by the loop and not by trusting any single LLM.
- Action-hash dedup at the supervisor level — same pattern from Lab 03, just lifted up.
- Step caps that compose across levels (`SUPERVISOR_MAX_STEPS = 6`, `WORKER_MAX_STEPS = 8`) — independent budgets, structured-error escalation.

The patterns from Path 01 transferred directly. The Lab 01 agent loop is the supervisor's loop. The Lab 02 tool-design principles govern how workers are described to the supervisor. The Lab 03 dedup + citation-tracking patterns extend across handoffs. **There was no new abstraction.**

That's the bet of this path: build the mechanism from first principles first; understand its failure modes; *then* adopt frameworks (CrewAI, AutoGen, LangGraph multi-agent) as ergonomic improvements, not as magic.

## Production readiness — out of scope here

For a real deployment you'd also want: structured logging at every handoff (worker name, payload hash, duration, status); per-worker token/cost budgets that the supervisor reads and respects; circuit breakers per worker (if researcher fails 3 times in 5 minutes, stop calling it for a while); async / parallel worker execution where the workers are genuinely independent; observability hooks for OTel-compatible tracing. The agent loop above is the foundation; production hardens its boundary layer.

## Next

- Take the [multi-agent fundamentals quiz](../../quizzes/multi-agent/multi-agent-fundamentals.md).
- The Path 03 v1 batch closes here. Future modules: agent debate (Module 2), plan-and-execute (Module 3), multi-agent RAG (Module 4), framework bridge (Module 5), multi-agent evaluation (Module 6).
- If you've also done Path 02, the future multi-agent RAG module will compose Lab 06-08's retrieval stack with this supervisor-worker pattern.